# Nimbus: reproducible GPU comparison and adapter export

Select **Runtime → Change runtime type → T4 GPU**, then run the cells in order. The public Qwen 2.5 3B checkpoint needs no Hugging Face token. This creates a new run; published historical results are never overwritten. The final ZIP includes both adapters, tokenizers, raw evaluations, training metadata, and a SHA-256 manifest. Download it before the runtime expires. Automatic scores still require answer review.


In [ ]:
import os, subprocess, sys
from pathlib import Path
def run_visible(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)

repo = Path("/content/llm-adaptation-comparison")
if not repo.exists():
    run_visible(["git", "clone", "--branch", "main", "https://github.com/Somama12/llm-adaptation-comparison.git", str(repo)])
os.chdir(repo)
run_visible([sys.executable, "-m", "pip", "install", "-r", "requirements-train.txt"])
print("Source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
import torch
assert torch.cuda.is_available(), "Select a T4 GPU runtime first"
print(torch.cuda.get_device_name())


## Run the complete workflow

This runs all four base-model conditions, trains Version 1, reloads it for evaluation, continues that adapter on Version 2, reloads it again, and packages both adapters. A failed command stops the workflow. The timestamp keeps previous results intact. Do not treat automatic scores as audited findings.


In [ ]:
from datetime import datetime, timezone
run_dir = Path("artifacts/runs") / datetime.now(timezone.utc).strftime("run-%Y%m%dT%H%M%S%fZ")
run_visible([sys.executable, "-m", "experiments.paired_run", "--run-dir", str(run_dir)])


In [ ]:
from google.colab import files
archive = run_dir.with_suffix(".zip")
run_visible([sys.executable, "-m", "experiments.export_run", "--verify", str(archive)])
files.download(str(archive))


## Verify the download on your computer

After downloading the ZIP, run `python -m experiments.export_run --verify /path/to/download.zip`. Keep the archive outside the temporary Colab runtime. See `docs/REPRODUCIBILITY.md` for adapter reload instructions and the distinction between historical evidence and this new reproduction.
